# Long Code Arena (Bug localization)
This is the benchmark for the Bug localization task as part of the Long Code Arena benchmark.
The dataset provides all the required components for evaluation of bug localization approaches in real project-level large-scale data collected from GitHub, including:

a. Bug issue description;

b. Repositories, from which the content can be extracted at the state of the commit SHA where the bug is reproducible;

c. List of files that should be changed in order to solve the bug;

d. Other additional data and metrics that can be useful in developing new approaches.

In [ ]:
import pandas as pd
from datasets import load_dataset
import ast   #Used to safely evaluate the string representation of a list

In [ ]:
def analyze_lca_dataset(config='py'):
    '''
    Loads and analyzes the Long Code Arena (LCA) bug localization dataset.

    Args:
        config (str): The language configuration to load ('py', 'java' or 'kt')
    '''
    print(f"----- Loading LCA Dataset for configuration: '{config}' -------")

    try:
        # Load all splits for the chosen configuration
        train_ds = load_dataset("/home/user/CS21D002_A_Eashaan_Rao/Research/PhD/Objective1/Resources/lca-bug-localization", config, split='train')
        dev_ds = load_dataset("/home/user/CS21D002_A_Eashaan_Rao/Research/PhD/Objective1/Resources/lca-bug-localization", config, split='dev')
        test_ds = load_dataset("/home/user/CS21D002_A_Eashaan_Rao/Research/PhD/Objective1/Resources/lca-bug-localization", config, split='test')

        # Convert to pandas DataFrame
        train_df = train_ds.to_pandas()
        dev_df = dev_ds.to_pandas()
        test_df = test_ds.to_pandas()
        return train_df, dev_df, test_df
    
    except Exception as e:
        print(f"An error occured: {e}")
        return None


In [ ]:
def find_unique_repos_data(df):
    # 1. Find Unique Repositories
    # Create a full repo for easier identification
    df['repo_full_name'] = df['repo_name'] + '/' + df['repo_name']
    unique_repositories = df['repo_full_name'].unique()

    print(f"Number of unique repositories: {len(unique_repositories)}\n")
    print(f"Unique bug ID: {len(df['id'].unique())}")
    print(f"Unique of Issue URL (bug report assuming): {len(df['issue_url'].unique())}")
    print(f"Unique of Pull Request URL (Ground files link assuming): {len(df['pull_url'].unique())}")
    # print("--- List of unique repositories ----")
    # for repo in sorted(unique_repositories):
    #     print(repo)


In [ ]:

def individual_records(df):
    # 2. Inspecting Individual records
    print("\n 3. Inspecting a few sample records...")

    # Combine issue title and body for the full description
    df['bug_description'] = df['issue_title'] + "\n" + df['issue_body']

    # the 'changed_files' column is a string that looks like a list. We need to parse it safely,
    # using a lmabda with an try-except block to handle potential errors
    df['ground_truth_files'] = df['changed_files'].apply(
        lambda x: ast.literal_eval(x) if isinstance(x, str) and x.startswith('[') else []
    )

    # Define the column we want to see
    columns_to_show = [
        'repo_full_name',
        'bug_description',
        'base_sha',
        'ground_truth_files'
    ]

    # Display the head of the selected columns
    # Using options to see the full contet
    pd.set_option('display.max_rows', 50)
    pd.set_option('display.max_colwidth', 150)
    print(df[columns_to_show].head(1))



### For Python dataset

In [ ]:
# for python repos
train_df, dev_df, test_df = analyze_lca_dataset(config='py')

In [ ]:
# Combine all splits into a single DataFrame
combined_df = pd.concat([train_df, dev_df, test_df], ignore_index=True).drop_duplicates(subset=['id'])

print(f"Total combined samples for py': {len(combined_df)}")
print(f"py dataset shape: {combined_df.shape}")
print(f"Columns of the dataset \n: {combined_df.columns}")

In [ ]:
# For unique repos in training, dev and test in python
# for training
print("****** For Training Set *******")
find_unique_repos_data(train_df)

# for dev
print("****** For Dev Set *******")
find_unique_repos_data(dev_df)

# for testing
print("****** For Testing Set *******")
find_unique_repos_data(test_df)

### For Java dataset

In [ ]:
# for java repos
train_df, dev_df, test_df = analyze_lca_dataset(config='java')

In [ ]:
# Combine all splits into a single DataFrame
combined_df = pd.concat([train_df, dev_df, test_df], ignore_index=True).drop_duplicates(subset=['id'])

print(f"Total combined samples for java': {len(combined_df)}")
print(f"java dataset shape: {combined_df.shape}")
print(f"Columns of the dataset \n: {combined_df.columns}")

In [ ]:
# For unique repos in training, dev and test in java"
# for training
print("****** For Training Set *******")
find_unique_repos_data(train_df)

# for dev
print("****** For Dev Set *******")
find_unique_repos_data(dev_df)

# for testing
print("****** For Testing Set *******")
find_unique_repos_data(test_df)

### For Kotlin Dataset

In [ ]:
# for kotlin repos
train_df, dev_df, test_df = analyze_lca_dataset(config='kt')

In [ ]:
# Combine all splits into a single DataFrame
combined_df = pd.concat([train_df, dev_df, test_df], ignore_index=True).drop_duplicates(subset=['id'])

print(f"Total combined samples for kotlin': {len(combined_df)}")
print(f"kotlin dataset shape: {combined_df.shape}")
print(f"Columns of the dataset \n: {combined_df.columns}")

In [ ]:
# For unique repos in training, dev and test in kotlin"
# for training
print("****** For Training Set *******")
find_unique_repos_data(train_df)

# for dev
print("****** For Dev Set *******")
find_unique_repos_data(dev_df)

# for testing
print("****** For Testing Set *******")
find_unique_repos_data(test_df)

# Repository - Number of Bug Reports per Repo - Duplicate Bug Reports

In [13]:
import pandas as pd

## Python Repos

In [15]:
# loads all train, dev, and test parquet files from a directory, combines them, and finds 
# unique repositories
lca_py_dev_file_path = '/home/user/CS21D002_A_Eashaan_Rao/Research/PhD/Objective1/Resources/lca-bug-localization/py/dev-00000-of-00001.parquet'
lca_py_train_file_path = '/home/user/CS21D002_A_Eashaan_Rao/Research/PhD/Objective1/Resources/lca-bug-localization/py/train-00000-of-00001.parquet'
lca_py_test_file_path = '/home/user/CS21D002_A_Eashaan_Rao/Research/PhD/Objective1/Resources/lca-bug-localization/py/test-00000-of-00001.parquet'

In [16]:
# load the data in pandas dataframe
lca_py_dev = pd.read_parquet(lca_py_dev_file_path)
print("Dev samples shape: ", lca_py_dev.shape)
lca_py_train = pd.read_parquet(lca_py_train_file_path)
print("Train samples shape: ", lca_py_train.shape)
lca_py_test = pd.read_parquet(lca_py_test_file_path)
print("Test samples shape: ", lca_py_test.shape)

Dev samples shape:  (4339, 41)
Train samples shape:  (4289, 41)
Test samples shape:  (50, 41)


In [19]:
print(lca_py_dev['changed_files'].apply(type).value_counts())
print(lca_py_dev['changed_files'].iloc[10])

changed_files
<class 'str'>    4339
Name: count, dtype: int64
['keras/engine/training.py']


In [ ]:
# For Dev splits
lca_py_dev['bug_report'] = lca_py_dev['issue_title'] + "\n" + lca_py_dev['issue_body']
lca_py_dev['repo'] = lca_py_dev['repo_owner'] + '/' + lca_py_dev['repo_name']

# For Train splits
lca_py_train['bug_report'] = lca_py_train['issue_title'] + "\n" + lca_py_train['issue_body']
lca_py_train['repo'] = lca_py_train['repo_owner'] + '/' + lca_py_train['repo_name']

# For Test splits
lca_py_test['bug_report'] = lca_py_test['issue_title'] + "\n" + lca_py_test['issue_body']
lca_py_test['repo'] = lca_py_test['repo_owner'] + '/' + lca_py_test['repo_name']

### For Dev Splits

Columns of the dataset 
: Index(['id', 'text_id', 'repo_owner', 'repo_name', 'issue_url', 'pull_url',
       'comment_url', 'links_count', 'link_keyword', 'issue_title',
       'issue_body', 'base_sha', 'head_sha', 'diff_url', 'diff',
       'changed_files', 'changed_files_exts', 'changed_files_count',
       'java_changed_files_count', 'kt_changed_files_count',
       'py_changed_files_count', 'code_changed_files_count',
       'repo_symbols_count', 'repo_tokens_count', 'repo_lines_count',
       'repo_files_without_tests_count', 'changed_symbols_count',
       'changed_tokens_count', 'changed_lines_count',
       'changed_files_without_tests_count', 'issue_symbols_count',
       'issue_words_count', 'issue_tokens_count', 'issue_lines_count',
       'issue_links_count', 'issue_code_blocks_count', 'pull_create_at',
       'repo_stars', 'repo_language', 'repo_languages', 'repo_license'],
      dtype='object')

In [ ]:
# Create the repository link from the issue_url
# This splits the URL and takes the first 5 parts (e.g., https://github.com/owner/repo)
lca_py_dev['repo_link'] = lca_py_dev['issue_url'].apply(lambda url: '/'.join(url.split('/')[:5]))


# Step 1: Count total bug reports per repo per different splits (train, test, dev)
# Group by 'repo' and count the number of instances
# # use .agg() to get multiple pieces of info from the grouped data
repo_info = lca_py_dev.groupby('repo').agg(
    Total_Bug_Reports = ('id', 'count'),
    repo_language = ('repo_language', 'first'),
    repo_link=('repo_link', 'first')
).reset_index()



# Step 2: Count duplicate bug reports per repo
# A duplicate is a non-unique 'problem statement' within a repo group.
# The number of duplocates is the total rows minus the number of unique rows.
duplicate_counts = lca_py_dev.groupby('repo').apply(
    lambda x: x.shape[0] - x['bug_report'].nunique()
).reset_index(name='Duplicate_Bug_Reports')

# Step 3: Combine and sort the results
# Merge two dataframes on the 'repo' column
results_df = pd.merge(repo_info, duplicate_counts, on='repo')

# Sort the final Dataframe in descending order by the number of bug reports
sorted_results = results_df.sort_values(by='Total_Bug_Reports', ascending=False)

# Reset index for cleaner presentation
sorted_results.reset_index(drop=True, inplace=True)

# Print only the repo_link column without the index ---
# print(sorted_results['repo_link'].to_string(index=False))

print("Dev Split Analysis")
print(sorted_results.to_string())

### For Train Splits

In [ ]:
# Step 1: Count total bug reports per repo per different splits (train, test, dev)
# Group by 'repo' and count the number of instances
bug_counts = lca_py_train.groupby('repo')['id'].count().reset_index()
bug_counts.rename(columns={'id': 'Total_Bug_Reports'}, inplace=True)

# Step 2: Count duplicate bug reports per repo
# A duplicate is a non-unique 'problem statement' within a repo group.
# The number of duplocates is the total rows minus the number of unique rows.
duplicate_counts = lca_py_train.groupby('repo').apply(
    lambda x: x.shape[0] - x['bug_report'].nunique()
).reset_index(name='Duplicate_Bug_Reports')

# Step 3: Combine and sort the results
# Merge two dataframes on the 'repo' column
results_df = pd.merge(bug_counts, duplicate_counts, on='repo')

# Sort the final Dataframe in descending order by the number of bug reports
sorted_results = results_df.sort_values(by='Total_Bug_Reports', ascending=False)

# Reset index for cleaner presentation
sorted_results.reset_index(drop=True, inplace=True)

print("Train Split Analysis")
print(sorted_results.to_string())

### For Test Splits

In [ ]:
# Step 1: Count total bug reports per repo per different splits (train, test, dev)
# Group by 'repo' and count the number of instances
bug_counts = lca_py_test.groupby('repo')['id'].count().reset_index()
bug_counts.rename(columns={'id': 'Total_Bug_Reports'}, inplace=True)

# Step 2: Count duplicate bug reports per repo
# A duplicate is a non-unique 'problem statement' within a repo group.
# The number of duplocates is the total rows minus the number of unique rows.
duplicate_counts = lca_py_test.groupby('repo').apply(
    lambda x: x.shape[0] - x['bug_report'].nunique()
).reset_index(name='Duplicate_Bug_Reports')

# Step 3: Combine and sort the results
# Merge two dataframes on the 'repo' column
results_df = pd.merge(bug_counts, duplicate_counts, on='repo')

# Sort the final Dataframe in descending order by the number of bug reports
sorted_results = results_df.sort_values(by='Total_Bug_Reports', ascending=False)

# Reset index for cleaner presentation
sorted_results.reset_index(drop=True, inplace=True)

print("Test Split Analysis")
print(sorted_results.to_string())

## For Java Repos

In [ ]:
# loads all train, dev, and test parquet files from a directory, combines them, and finds 
# unique repositories
lca_java_dev_file_path = '/home/user/CS21D002_A_Eashaan_Rao/Research/PhD/Objective1/Resources/lca-bug-localization/java/dev-00000-of-00001.parquet'
lca_java_train_file_path = '/home/user/CS21D002_A_Eashaan_Rao/Research/PhD/Objective1/Resources/lca-bug-localization/java/train-00000-of-00001.parquet'
lca_java_test_file_path = '/home/user/CS21D002_A_Eashaan_Rao/Research/PhD/Objective1/Resources/lca-bug-localization/java/test-00000-of-00001.parquet'

In [ ]:
# load the data in pandas dataframe
lca_java_dev = pd.read_parquet(lca_java_dev_file_path)
print("Dev samples shape: ", lca_java_dev.shape)
lca_java_train = pd.read_parquet(lca_java_train_file_path)
print("Train samples shape: ", lca_java_train.shape)
lca_java_test = pd.read_parquet(lca_java_test_file_path)
print("Test samples shape: ", lca_java_test.shape)

In [ ]:
# For Dev splits
lca_java_dev['bug_report'] = lca_java_dev['issue_title'] + "\n" + lca_java_dev['issue_body']
lca_java_dev['repo'] = lca_java_dev['repo_owner'] + '/' + lca_java_dev['repo_name']

# For Train splits
lca_java_train['bug_report'] = lca_java_train['issue_title'] + "\n" + lca_java_train['issue_body']
lca_java_train['repo'] = lca_java_train['repo_owner'] + '/' + lca_java_train['repo_name']

# For Test splits
lca_java_test['bug_report'] = lca_java_test['issue_title'] + "\n" + lca_java_test['issue_body']
lca_java_test['repo'] = lca_java_test['repo_owner'] + '/' + lca_java_test['repo_name']

### For Dev Split

In [ ]:
# Step 1: Count total bug reports per repo per different splits (train, test, dev)
# Group by 'repo' and count the number of instances
bug_counts = lca_java_dev.groupby('repo')['id'].count().reset_index()
bug_counts.rename(columns={'id': 'Total_Bug_Reports'}, inplace=True)

# Step 2: Count duplicate bug reports per repo
# A duplicate is a non-unique 'problem statement' within a repo group.
# The number of duplocates is the total rows minus the number of unique rows.
duplicate_counts = lca_java_dev.groupby('repo').apply(
    lambda x: x.shape[0] - x['bug_report'].nunique()
).reset_index(name='Duplicate_Bug_Reports')

# Step 3: Combine and sort the results
# Merge two dataframes on the 'repo' column
results_df = pd.merge(bug_counts, duplicate_counts, on='repo')

# Sort the final Dataframe in descending order by the number of bug reports
sorted_results = results_df.sort_values(by='Total_Bug_Reports', ascending=False)

# Reset index for cleaner presentation
sorted_results.reset_index(drop=True, inplace=True)

print("Dev Split Analysis")
print(sorted_results.to_string())

### For Train Split

In [ ]:
# Step 1: Count total bug reports per repo per different splits (train, test, dev)
# Group by 'repo' and count the number of instances
bug_counts = lca_java_train.groupby('repo')['id'].count().reset_index()
bug_counts.rename(columns={'id': 'Total_Bug_Reports'}, inplace=True)

# Step 2: Count duplicate bug reports per repo
# A duplicate is a non-unique 'problem statement' within a repo group.
# The number of duplocates is the total rows minus the number of unique rows.
duplicate_counts = lca_java_train.groupby('repo').apply(
    lambda x: x.shape[0] - x['bug_report'].nunique()
).reset_index(name='Duplicate_Bug_Reports')

# Step 3: Combine and sort the results
# Merge two dataframes on the 'repo' column
results_df = pd.merge(bug_counts, duplicate_counts, on='repo')

# Sort the final Dataframe in descending order by the number of bug reports
sorted_results = results_df.sort_values(by='Total_Bug_Reports', ascending=False)

# Reset index for cleaner presentation
sorted_results.reset_index(drop=True, inplace=True)

print("Train Split Analysis")
print(sorted_results.to_string())

### For Test Split

In [ ]:
# Step 1: Count total bug reports per repo per different splits (train, test, dev)
# Group by 'repo' and count the number of instances
bug_counts = lca_java_test.groupby('repo')['id'].count().reset_index()
bug_counts.rename(columns={'id': 'Total_Bug_Reports'}, inplace=True)

# Step 2: Count duplicate bug reports per repo
# A duplicate is a non-unique 'problem statement' within a repo group.
# The number of duplocates is the total rows minus the number of unique rows.
duplicate_counts = lca_java_test.groupby('repo').apply(
    lambda x: x.shape[0] - x['bug_report'].nunique()
).reset_index(name='Duplicate_Bug_Reports')

# Step 3: Combine and sort the results
# Merge two dataframes on the 'repo' column
results_df = pd.merge(bug_counts, duplicate_counts, on='repo')

# Sort the final Dataframe in descending order by the number of bug reports
sorted_results = results_df.sort_values(by='Total_Bug_Reports', ascending=False)

# Reset index for cleaner presentation
sorted_results.reset_index(drop=True, inplace=True)

print("Test Split Analysis")
print(sorted_results.to_string())

## For Kotlin Repos

In [ ]:
# loads all train, dev, and test parquet files from a directory, combines them, and finds 
# unique repositories
lca_kt_dev_file_path = '/home/user/CS21D002_A_Eashaan_Rao/Research/PhD/Objective1/Resources/lca-bug-localization/kt/dev-00000-of-00001.parquet'
lca_kt_train_file_path = '/home/user/CS21D002_A_Eashaan_Rao/Research/PhD/Objective1/Resources/lca-bug-localization/kt/train-00000-of-00001.parquet'
lca_kt_test_file_path = '/home/user/CS21D002_A_Eashaan_Rao/Research/PhD/Objective1/Resources/lca-bug-localization/kt/test-00000-of-00001.parquet'

In [ ]:
# load the data in pandas dataframe
lca_kt_dev = pd.read_parquet(lca_kt_dev_file_path)
print("Dev samples shape: ", lca_kt_dev.shape)
lca_kt_train = pd.read_parquet(lca_kt_train_file_path)
print("Train samples shape: ", lca_kt_train.shape)
lca_kt_test = pd.read_parquet(lca_kt_test_file_path)
print("Test samples shape: ", lca_kt_test.shape)

In [ ]:
# For Dev splits
lca_kt_dev['bug_report'] = lca_kt_dev['issue_title'] + "\n" + lca_kt_dev['issue_body']
lca_kt_dev['repo'] = lca_kt_dev['repo_owner'] + '/' + lca_kt_dev['repo_name']

# For Train splits
lca_kt_train['bug_report'] = lca_kt_train['issue_title'] + "\n" + lca_kt_train['issue_body']
lca_kt_train['repo'] = lca_kt_train['repo_owner'] + '/' + lca_kt_train['repo_name']

# For Test splits
lca_kt_test['bug_report'] = lca_kt_test['issue_title'] + "\n" + lca_kt_test['issue_body']
lca_kt_test['repo'] = lca_kt_test['repo_owner'] + '/' + lca_kt_test['repo_name']

### For Dev Split

In [ ]:
# Step 1: Count total bug reports per repo per different splits (train, test, dev)
# Group by 'repo' and count the number of instances
bug_counts = lca_kt_dev.groupby('repo')['id'].count().reset_index()
bug_counts.rename(columns={'id': 'Total_Bug_Reports'}, inplace=True)

# Step 2: Count duplicate bug reports per repo
# A duplicate is a non-unique 'problem statement' within a repo group.
# The number of duplocates is the total rows minus the number of unique rows.
duplicate_counts = lca_kt_dev.groupby('repo').apply(
    lambda x: x.shape[0] - x['bug_report'].nunique()
).reset_index(name='Duplicate_Bug_Reports')

# Step 3: Combine and sort the results
# Merge two dataframes on the 'repo' column
results_df = pd.merge(bug_counts, duplicate_counts, on='repo')

# Sort the final Dataframe in descending order by the number of bug reports
sorted_results = results_df.sort_values(by='Total_Bug_Reports', ascending=False)

# Reset index for cleaner presentation
sorted_results.reset_index(drop=True, inplace=True)

print("Dev Split Analysis")
print(sorted_results.to_string())

### For Train Split

In [ ]:
# Step 1: Count total bug reports per repo per different splits (train, test, dev)
# Group by 'repo' and count the number of instances
bug_counts = lca_kt_train.groupby('repo')['id'].count().reset_index()
bug_counts.rename(columns={'id': 'Total_Bug_Reports'}, inplace=True)

# Step 2: Count duplicate bug reports per repo
# A duplicate is a non-unique 'problem statement' within a repo group.
# The number of duplocates is the total rows minus the number of unique rows.
duplicate_counts = lca_kt_train.groupby('repo').apply(
    lambda x: x.shape[0] - x['bug_report'].nunique()
).reset_index(name='Duplicate_Bug_Reports')

# Step 3: Combine and sort the results
# Merge two dataframes on the 'repo' column
results_df = pd.merge(bug_counts, duplicate_counts, on='repo')

# Sort the final Dataframe in descending order by the number of bug reports
sorted_results = results_df.sort_values(by='Total_Bug_Reports', ascending=False)

# Reset index for cleaner presentation
sorted_results.reset_index(drop=True, inplace=True)

print("Train Split Analysis")
print(sorted_results.to_string())

### For Test Split

In [ ]:
# Step 1: Count total bug reports per repo per different splits (train, test, dev)
# Group by 'repo' and count the number of instances
bug_counts = lca_kt_test.groupby('repo')['id'].count().reset_index()
bug_counts.rename(columns={'id': 'Total_Bug_Reports'}, inplace=True)

# Step 2: Count duplicate bug reports per repo
# A duplicate is a non-unique 'problem statement' within a repo group.
# The number of duplocates is the total rows minus the number of unique rows.
duplicate_counts = lca_kt_test.groupby('repo').apply(
    lambda x: x.shape[0] - x['bug_report'].nunique()
).reset_index(name='Duplicate_Bug_Reports')

# Step 3: Combine and sort the results
# Merge two dataframes on the 'repo' column
results_df = pd.merge(bug_counts, duplicate_counts, on='repo')

# Sort the final Dataframe in descending order by the number of bug reports
sorted_results = results_df.sort_values(by='Total_Bug_Reports', ascending=False)

# Reset index for cleaner presentation
sorted_results.reset_index(drop=True, inplace=True)

print("Test Split Analysis")
print(sorted_results.to_string())

## Finding Total Unique Bug Reports

In [ ]:
# Load the dev split of Python, Java and Kotlin repos
lca_py_dev_file_path = '/home/user/CS21D002_A_Eashaan_Rao/Research/PhD/Objective1/Resources/lca-bug-localization/py/dev-00000-of-00001.parquet'
lca_kt_dev_file_path = '/home/user/CS21D002_A_Eashaan_Rao/Research/PhD/Objective1/Resources/lca-bug-localization/kt/dev-00000-of-00001.parquet'
lca_java_dev_file_path = '/home/user/CS21D002_A_Eashaan_Rao/Research/PhD/Objective1/Resources/lca-bug-localization/java/dev-00000-of-00001.parquet'

In [ ]:
lca_py_dev = pd.read_parquet(lca_py_dev_file_path)
print("Python Dev samples shape: ", lca_py_dev.shape)
lca_kt_dev = pd.read_parquet(lca_kt_dev_file_path)
print("Kotlin Dev samples shape: ", lca_kt_dev.shape)
lca_java_dev = pd.read_parquet(lca_java_dev_file_path)
print("Java Dev samples shape: ", lca_java_dev.shape)

In [ ]:
# Creating new fields for combined bug reports and repo 
lca_py_dev['bug_report'] = lca_py_dev['issue_title'] + "\n" + lca_py_dev['issue_body']
lca_py_dev['repo'] = lca_py_dev['repo_owner'] + '/' + lca_py_dev['repo_name']

lca_java_dev['bug_report'] = lca_java_dev['issue_title'] + "\n" + lca_java_dev['issue_body']
lca_java_dev['repo'] = lca_java_dev['repo_owner'] + '/' + lca_java_dev['repo_name']

lca_kt_dev['bug_report'] = lca_kt_dev['issue_title'] + "\n" + lca_kt_dev['issue_body']
lca_kt_dev['repo'] = lca_kt_dev['repo_owner'] + '/' + lca_kt_dev['repo_name']

In [ ]:
# Combine the python, java, kotlin dev dataframes into on
lca_dev = pd.concat([lca_py_dev, lca_java_dev, lca_kt_dev], ignore_index=True)

In [ ]:
# Create the repository link from the issue_url
# This splits the URL and takes the first 5 parts (e.g., https://github.com/owner/repo)
lca_dev['repo_link'] = lca_dev['issue_url'].apply(
    lambda url: '/'.join(url.split('/')[:5]) if isinstance(url, str) else ''
    )


# Count total bug reports per repo 
# Group by 'repo' and count the number of instances
# We count total report ('id') and the number of unique fixes ('pull_url')
repo_analysis = lca_dev.groupby('repo').agg(
    Total_Bug_Reports = ('id', 'count'),
    Unique_Fix_Count = ('pull_url', 'nunique'),
    repo_language = ('repo_language', 'first'),
    repo_link=('repo_link', 'first')
).reset_index()



# Count duplicate and unique bug report columns from the aggregated data
repo_analysis['Duplicate_Bug_Reports'] = repo_analysis['Total_Bug_Reports'] - repo_analysis['Unique_Fix_Count']
repo_analysis['Unique_Bug_Reports'] = repo_analysis['Total_Bug_Reports'] - repo_analysis['Duplicate_Bug_Reports']

# Finalize the DataFrame for presentation.S elect and reorder the columsn for the final output
final_columns = [
    'repo', 'repo_link', 'repo_language', 'Total_Bug_Reports', 'Duplicate_Bug_Reports', 'Unique_Bug_Reports'
]

sorted_results = repo_analysis[final_columns].sort_values(by='Total_Bug_Reports', ascending=False).reset_index(drop=True)

print("LCA combined Analysis (py, java, kt)")
print(sorted_results.to_string())

In [ ]:
sorted_results.to_csv("lca_analysis_rd1.csv", index=False)


## Finding ground truth files extensions from all repos for java, kotlin, python

In [ ]:
def analyze_lca_unique_extensions(lca_dev):
    '''
    finds the unique file extensions for each language in the Long Code Arena dataset by 
    parsing the 'changed_files_exts' column

    Args:
        lca_combined_df (pd.DataFrame): the combined dataframe from the LCA dataset using dev split
    '''
    print("Analyzing Unique File extensions in long code arena")

    if 'repo_language' not in lca_dev.columns or 'changed_files_exts' not in lca_dev.columns:
        print("Error: 'repo_language' and 'changed_files_exts' columns must be present.")
        return
    
    # Get the list of unique languages in the dataset
    languages = lca_dev['repo_language'].unique()

    # Analyze each language separately
    for lang in sorted(languages):
        print(f"Analysis for Language: {lang}")

        # Filter the DataFrame for the current language
        lang_df = lca_dev[lca_dev['repo_language'] == lang].copy()

        # Safely parse the string representation of a dictionary into an actual dictionary
        lang_df['ext_dict'] = lang_df['changed_files_exts'].apply(
            lambda x: ast.literal_eval(x) if isinstance(x,str) and x.startswith('{') else {}
        )

        # Extract the keys (the extensions) from each dictionary
        lang_df['extensions_list'] = lang_df['ext_dict'].apply(lambda d: list(d.keys()))

        # Explode the Dataframe to have one row per extension
        all_extensions_df = lang_df.explode('extensions_list')

        # Find the unique extensions and sort them
        unique_extensions = sorted(all_extensions_df['extensions_list'].dropna().unique())

        print("Found the following unique file extensions:")
        print(unique_extensions)



In [ ]:
analyze_lca_unique_extensions(lca_dev)

# MetaData Extraction
Set of methods to calculate all ten metadata for selected projects from LCA

1. LOC
2. age_years
3. median_bug_year
4. num_authors
5. num_commits
6. num_dependencies
7. polyglot_index
8. bug_density
9. code_complexity
10. bug_report_verbosity

In [1]:
# Import libraries
import pandas as pd
import os
import re
import git
import subprocess
from datetime import datetime
from tqdm import tqdm
import xml.etree.ElementTree as ET
import tempfile 

## Configuration

In [2]:
# 1. Path to the input CSV file
# Must have columns: 'repo_name', 'language', 'total_unique_bug_report'
INPUT_CSV_PATH = "/home/user/CS21D002_A_Eashaan_Rao/Research/PhD/Objective1/benchmark_dataset_analysis/lca_metadata_input.csv"

# 2. Path to the main LCA dataset file (to get commite and date info)
lca_py_dev_file_path = '/home/user/CS21D002_A_Eashaan_Rao/Research/PhD/Objective1/Resources/lca-bug-localization/py/dev-00000-of-00001.parquet'
lca_java_dev_file_path = '/home/user/CS21D002_A_Eashaan_Rao/Research/PhD/Objective1/Resources/lca-bug-localization/java/dev-00000-of-00001.parquet'
lca_kt_dev_file_path = '/home/user/CS21D002_A_Eashaan_Rao/Research/PhD/Objective1/Resources/lca-bug-localization/kt/dev-00000-of-00001.parquet'

# Load dev split into pandas dataframe
lca_py_dev = pd.read_parquet(lca_py_dev_file_path)
lca_java_dev = pd.read_parquet(lca_java_dev_file_path)
lca_kt_dev = pd.read_parquet(lca_kt_dev_file_path)

lca_py_dev['bug_report'] = lca_py_dev['issue_title'] + "\n" + lca_py_dev['issue_body']
lca_py_dev['repo'] = lca_py_dev['repo_owner'] + '/' + lca_py_dev['repo_name']
lca_java_dev['bug_report'] = lca_java_dev['issue_title'] + "\n" + lca_java_dev['issue_body']
lca_java_dev['repo'] = lca_java_dev['repo_owner'] + '/' + lca_java_dev['repo_name']
lca_kt_dev['bug_report'] = lca_kt_dev['issue_title'] + "\n" + lca_kt_dev['issue_body']
lca_kt_dev['repo'] = lca_kt_dev['repo_owner'] + '/' + lca_kt_dev['repo_name']

# Combine the python, java, kotlin dev dataframes into on
lca_df = pd.concat([lca_py_dev, lca_java_dev, lca_kt_dev], ignore_index=True)

# 3. Path to the parent directory where repos are cloned
CLONE_DIR = '/home/user/CS21D002_A_Eashaan_Rao/Research/PhD/Objective1/temp_repos/'

# 4. Path for the final output CqSV file.
OUTPUT_CSV_PATH = '/home/user/CS21D002_A_Eashaan_Rao/Research/PhD/Objective1/benchmark_dataset_analysis/lca_metadata.csv'

# 5. Extensions for Polyglot Index Calculation
LANGUAGE_EXTENSIONS = {
    'c++': ['.c', '.cc', '.cmake', '.cpp', '.cxx', '.h', '.hh', '.hpp', '.hxx', '.in', '.json', '.make', '.py', '.sh', '.xml'],
    'go': ['.go', '.json', '.proto', '.sh', '.yaml', '.yml'],
    'java': ['.gradle', '.groovy', '.java', '.json', '.properties', '.xml', '.yml', '.yaml'],
    'javascript': ['.css', '.html', '.js', '.json', '.jsx', '.mjs', '.scss', '.sh', '.ts', '.tsx', '.yaml', '.yml'],
    'kotlin': ['.gradle', '.json', '.kt', '.kts', '.properties', '.xml', '.yaml', '.yml'],
    'python': ['.bash', '.cfg', '.in', '.ini', '.json', '.py', '.sh', '.toml', '.yaml', '.yml']
}
PRIMARY_EXTENSIONS = {
    'python': ['.py'],
    'java': ['.java'],
    'kotlin': ['.kt'],
    'c++': ['.c', '.cc', '.cpp', '.cxx', '.h', '.hh', '.hpp', '.hxx'],
    'go': ['.go'],
    'javascript': ['.js', '.jsx', '.mjs', '.ts', '.tsx']
}

## Helper Methods

In [3]:
# aiming to find correct bug report date and timelines.
def get_commit_date(repo_object, commit_hash):
    '''
    Helper function to safely get a commit's authored date.
    '''
    try:
        # Get the timezone-aware datetime object
        dt_object = repo_object.commit(commit_hash).authored_datetime
        # Convert to UTC for consistency
        return dt_object.astimezone(pd.Timestamp("2000-01-01").tz)
    except Exception:
        return pd.NaT

In [4]:
# Group by repository to process one repo at a time - the purpose to find correct commit authored date 
# (continuing from above cell)
# Load input files
try:
    selected_repos_df = pd.read_csv(INPUT_CSV_PATH)
    # print(selected_repos_df)
except FileNotFoundError as e:
    print(f"Error: Input file not found. {e}")
    exit(0)

# Initialize the new column with a timezone-aware NaT
# This solves the FutureWarning by making the dtypes compatible from the start.
lca_df['report_date'] = pd.to_datetime(pd.NaT, utc=True)


for _, row in tqdm(selected_repos_df.iterrows(), total=len(selected_repos_df), desc="Processing Repos for dates"):
    repo_name = row['repo_name']
    language = row['language']
    repo_path = os.path.join(CLONE_DIR, language.lower(), repo_name.replace('/', '_'))
    
    tqdm.write(f"Working on repo: {repo_name}")
    if not os.path.exists(repo_path):
        print(f"Warning: Clone repo not found for {repo_name} at {repo_path}. Skipping.")
        continue

    # Filter the main Dataframe to get the specific rows for this repo
    # Using .index is crucial for the assignment step later
    repo_data_indices = lca_df[lca_df['repo'] == repo_name].index

    if repo_data_indices.empty:
        print(f"Warning: No data found for {repo_name} in LCA main file. Skipping.")
        continue
    
    try:
        repo = git.Repo(repo_path)

        # Get the base_sha vales for the current repo's data
        base_shas = lca_df.loc[repo_data_indices, 'base_sha']
        # print(base_shas)

        # Apply the date extraction function
        dates_for_group = base_shas.apply(lambda sha: get_commit_date(repo, sha))

        # --- FIX 1: Ensure the Series has the correct dtype before assignment ---
        # This also helps prevent the FutureWarning
        dates_for_group = pd.to_datetime(dates_for_group, utc=True)
        
        # Assign the calculated dates back to the correct rows
        # The FutureWarning should now be resolved by the initialization fix
        lca_df.loc[repo_data_indices, 'report_date'] = dates_for_group

    except Exception as e:
        tqdm.write(f"Could not process repo {repo_name}. Error: {e}")

print("Pre-processing complete. 'report_date' column has been updated for selected repos")
print(lca_df[lca_df['repo_name'] == selected_repos_df['repo_name'].iloc[0]][['repo_name', 'report_date']].head())


Processing Repos for dates:   0%|          | 0/13 [00:00<?, ?it/s]

Working on repo: quarkusio/quarkus


/tmp/ipykernel_46311/230890354.py:50: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '<DatetimeArray>
['2023-05-24 14:39:04+00:00', '2023-05-21 11:32:06+00:00',
 '2023-05-19 13:54:00+00:00', '2023-05-18 08:12:11+00:00',
 '2023-05-17 14:17:34+00:00', '2023-05-17 09:09:32+00:00',
 '2023-05-17 09:09:32+00:00', '2023-05-18 08:12:11+00:00',
 '2023-05-22 17:02:42+00:00', '2023-05-15 12:08:39+00:00',
 ...
 '2023-07-04 05:51:03+00:00', '2023-07-03 15:00:10+00:00',
 '2023-07-06 17:45:50+00:00', '2023-07-17 16:42:50+00:00',
 '2023-07-21 16:51:34+00:00', '2023-07-20 11:10:11+00:00',
 '2023-07-19 11:36:57+00:00', '2023-07-17 08:48:20+00:00',
 '2020-08-13 14:25:02+00:00', '2021-11-09 16:06:33+00:00']
Length: 957, dtype: datetime64[ns, UTC]' has dtype incompatible with datetime64[ns], please explicitly cast to a compatible dtype first.
  lca_df.loc[repo_data_indices, 'report_date'] = dates_for_group
Processing Repos for

Working on repo: hannah-sten/texify-idea
Working on repo: horizontalsystems/unstoppable-wallet-android
Working on repo: intellij-rust/intellij-rust
Working on repo: jwstegemann/fritz2
Working on repo: square/anvil
Working on repo: square/kotlinpoet


Processing Repos for dates:  69%|██████▉   | 9/13 [00:00<00:00, 18.74it/s]

Working on repo: square/leakcanary
Working on repo: square/okhttp
Working on repo: thundernest/k-9
Working on repo: dmwm/wmcore
Working on repo: iterative/dvc


Processing Repos for dates: 100%|██████████| 13/13 [00:00<00:00, 13.98it/s]

Working on repo: rucio/rucio
Pre-processing complete. 'report_date' column has been updated for selected repos
Empty DataFrame
Columns: [repo_name, report_date]
Index: []


In [5]:
# Pre-requisite: Assume 'lca_df' is loaded and has the 'report_date' column
print(" Verifying the corrected 'report_date' column")

# The 'report_date' column should already be in datetime formate for your pre-processing
# we will just ensure it, coercing any potential errors that might still exist
# lca_df['report_date'] = pd.to_datetime(lca_df['report_date'], errors='coerce')
lca_df['report_date'] = pd.to_datetime(lca_df['report_date'])

# 1. Show basic statistics of the corrected dates
print("Overall corrected date statistics:")
# Filter out any NaT values for accurate stats
valid_dates = lca_df['report_date'].dropna()
if not valid_dates.empty:
    print(f" Earliest Date: {valid_dates.min()}")
    print(f" Latest Date: {valid_dates.max()}")
    print(f" Median Date: {valid_dates.median()}")
else:
    print(" No valid dates found in the 'report_date' column.")

# 2. Show the distribution of bug reports by year
print("Distribution of Bug Reports per Year (top 15):")
print(valid_dates.dt.year.value_counts().sort_index(ascending=False).head(15).to_string())

# 3. Isolate and check if any '1970' dates remain
print("Checking for any remaining anomalous '1970' dates")
problematic_rows = lca_df[lca_df['report_date'].dt.year == 1970]

if not problematic_rows.empty:
    print(f"Warning: Found {len(problematic_rows)} entries that still have a '1970' year")
else:
    print("Success: No entries with a '1970' year were found in the 'report_date' column.")

 Verifying the corrected 'report_date' column
Overall corrected date statistics:
 Earliest Date: 2012-08-01 16:27:21+00:00
 Latest Date: 2023-08-17 18:53:04+00:00
 Median Date: 2020-10-14 03:02:57.500000+00:00
Distribution of Bug Reports per Year (top 15):
report_date
2023    205
2022    400
2021    494
2020    472
2019    490
2018    197
2017     30
2016     32
2015      8
2014     23
2013      6
2012      1
Checking for any remaining anomalous '1970' dates
Success: No entries with a '1970' year were found in the 'report_date' column.


In [6]:
# Check the pull_create_at timestamp
# 1. Convert the column to datetime, turning any errors into 'NaT' (Not a Time)
# this prevents the script from crashing and lets us see the problematic data
lca_df['parsed_date'] = pd.to_datetime(lca_df['pull_create_at'], errors ='coerce')

# 2. Show basic statistics of the parsed dates
print("Overall Data Statistics...")
print(f" Earliest Date: {lca_df['parsed_date'].min()}")
print(f" Latest Date: {lca_df['parsed_date'].max()}")
print(f" median Date: {lca_df['parsed_date'].median()}")

# 3. Show the distribution of bug reports by year
print("Distributon of Bug Report per year (top 15)")
print(lca_df['parsed_date'].dt.year.value_counts().sort_index(ascending=False).head(15).to_string())

# 4. Isolate and show any rows that were parsed as 1970
print(" Checking for anomalous '1970' dates")
problematic_rows = lca_df[lca_df['parsed_date'].dt.year == 1970]

if not problematic_rows.empty:
    print(f"Found {len(problematic_rows)} entries with a '1970' year")
else:
    print("No entries with a '1970' year were found after parsing.")


Overall Data Statistics...
 Earliest Date: 1970-01-01 00:22:23
 Latest Date: 1970-01-01 00:28:12
 median Date: 1970-01-01 00:26:37
Distributon of Bug Report per year (top 15)
parsed_date
1970    7479
 Checking for anomalous '1970' dates
Found 7479 entries with a '1970' year


In [7]:
def count_lines_in_file(file_path):
    '''
    Counts lines in a file, handling encoding errors.
    '''
    try:
        with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
            return len(f.readlines())
    except Exception:
        return 0

In [8]:
def calculate_loc_and_polyglot(repo_path, declared_language):
    '''
    Calculates LoC and polyglot index. Auto-detects the actual primary language in the snapshot to handle
    migrations
    '''
    loc_by_lang = {lang: 0 for lang in PRIMARY_EXTENSIONS.keys()}
    loc_relevant = 0

    # first, calculate LoC for each potential primary language
    for root, _, files in os.walk(repo_path):
        for file in files:
            for lang, exts in PRIMARY_EXTENSIONS.items():
                if file.endswith(tuple(exts)):
                    loc_by_lang[lang] += count_lines_in_file(os.path.join(root, file))

    # Auto-detect the language with the most LoC in this snapshot
    actual_primary_languge = max(loc_by_lang, key=loc_by_lang.get) if loc_by_lang else declared_language
    loc_primary = loc_by_lang.get(actual_primary_languge, 0)

    # Now, calculate total relevant LoC based on the DECLARED ecosystem
    relevant_exts = tuple(LANGUAGE_EXTENSIONS.get(declared_language.lower(), []))

    for root, _, files in os.walk(repo_path):
        for file in files:
            if file.endswith(relevant_exts):
                loc_relevant += count_lines_in_file(os.path.join(root, file))
                
    polyglot_index = (loc_primary / loc_relevant) if loc_relevant > 0 else 0
    return loc_relevant, polyglot_index

In [9]:
def count_dependencies(repo_path, language):
    """
    Calculates a proxy for dependency complexity by summing the Lines of Code (LoC)
    of standard dependency files for the primary language.
    """
    loc_count = 0
    lang = language.lower()

    dependency_files = []
    if lang == 'python':
        dependency_files = ['requirements.txt', 'pyproject.toml']
    elif lang in ['java', 'kotlin']:
        dependency_files = ['pom.xml', 'build.gradle', 'build.gradle.kts']
    elif lang == 'c++':
        dependency_files = ['CMakeLists.txt', 'Makefile'] # Example for C++
    elif lang == 'javascript':
        dependency_files = ['package.json'] # Example for JS
    elif lang == 'go':
        dependency_files = ['go.mod'] # Example for Go

    for root, _, files in os.walk(repo_path):
        for file in files:
            if file in dependency_files:
                file_path = os.path.join(root, file)
                # Simply add the number of lines in the file to the count
                loc_count += count_lines_in_file(file_path)
    
    return loc_count

In [10]:
def calculate_average_complexity(repo_path, language):
    '''
    Calculates average cyclomatic complexity using the 'lizard' tool.
    We store the ouput of lizard to a temporary file to handle large outputs reliably.
    '''
    # Create a temp file to stroe the lizard output
    with tempfile.NamedTemporaryFile(mode='w+', delete=False, suffix='.txt', encoding='utf-8') as temp_out:
        temp_filename = temp_out.name

    try:
        # lizard expects 'c++' to be written as 'cpp'
        lang_for_lizard = 'cpp' if language.lower() == 'c++' else language.lower()
        # print("Language for lizard: ", lang_for_lizard)

        # Redirect stdout to the temp file
        # We run the command and tell it to write its output directly to our temp file
        result = subprocess.run(
            ['lizard', '-i', '0', repo_path], # another command ['lizard', '-l', 'lang_for_lizard', repo_path]
            stdout = open(temp_filename, 'w',  encoding='utf-8'), # Write stdout to the temp file
            stderr=subprocess.PIPE, # Still capture any errors in memory 
            check=False, text=True
        )

        # We can still manually check the return code if we want to log detailed errors
        if result.returncode != 0:
            print(f"\nWarning: Lizard finished with a non-zero exit code ({result.returncode}) for {repo_path}. This usually indicates warnings were found. Continuing to parse output.")
            # We don't return here, because the output file is likely still valid.

        # Read the results back from the file
        with open(temp_filename, 'r', encoding='utf-8') as f:
            lizard_output = f.read()

        # Get all non-empty lines from the output
        lines = [line for line in lizard_output.strip().splitlines()]

        # the summary data is on the second to last line
        if len(lines) >=3 :
            # target the line with the numbers (the last non-empty line)
            summary_line = lines[-1]

            # Split the line by whitespace
            values = summary_line.split()

            if len(values) >= 3:
                # The Avg CCN is the 3rd value (index 2)
                avg_ccn = float(values[2])
                return avg_ccn

        # Find the summary line in the output
        # If we reach here, the summary line was not found or was malformed
        print(f"\nWarning: Could not parse lizard summary for {repo_path}.")
        return 0.0
        
    except FileNotFoundError:
        # This error is critical, so we print it once and then it will return 0 for others.
        print("\nERROR: 'lizard' command not found. Please install it with 'pip install lizard'.")
        return 0.0
    except subprocess.CalledProcessError as e:
        print(f"\n Lizard command failed for {repo_path}. Stderr: {e.stderr}")
        return 0.0
    except (IndexError, ValueError) as e:
        print(f"\nFailed to extract complexity value from summary line for {repo_path}. Error: {e}")
        return 0.0
    except Exception as e:
        print(f"\nAn unexpected error occurred in calculate_average_complexity: {e}")
        return 0.0
    finally:
        if os.path.exists(temp_filename):
            os.remove(temp_filename)

In [11]:
def categorize_by_tercile(group):
    '''
    Assigns size categories ('small', 'medium', 'large') based on LoC.
    '''
    tercile_1 = group['LoC'].quantile(1/3)
    tercile_2 = group['LoC'].quantile(2/3)

    def assign_category(loc):
        if loc <= tercile_1: return 'small'
        elif loc <= tercile_2: return 'medium'
        else: return 'high'
    group['project_size'] = group['LoC'].apply(assign_category)
    return group

## Main code

In [12]:
print("Starting metadata extraction for LCA Dataset...")

# Load input files
try:
    selected_repos_df = pd.read_csv(INPUT_CSV_PATH)
    # print(selected_repos_df)
except FileNotFoundError as e:
    print(f"Error: Input file not found. {e}")
    exit(0)

results = []

for _, row in tqdm(selected_repos_df.iterrows(), total=len(selected_repos_df), desc="Processing Repos"):
    repo_name = row['repo_name']
    language = row['language']
    unique_bugs = row['total_unique_bug_report']

    repo_path = os.path.join(CLONE_DIR, language.lower(), repo_name.replace('/', '_'))
    print(f"Calculating meta data for repo: {repo_name}")
    if not os.path.exists(repo_path):
        print(f"Warning: Clone repo not found for {repo_name} at {repo_path}. Skipping.")
        continue

    repo_lca_data = lca_df[lca_df['repo'] == repo_name].copy()
    if repo_lca_data.empty:
        print(f"Warning: No data found for {repo_name} in LCA main file. Skipping.")
        continue

    # Get the snapshot commit from the latest bug report
    latest_bug = repo_lca_data.sort_values(by='report_date', ascending=False).iloc[0]
    snapshot_commit = latest_bug['base_sha']

    try:
        repo = git.Repo(repo_path)
        repo.git.checkout(snapshot_commit, f=True)

        # *****************
        # Calculate Metrics
        # *****************

        # c) Age
        # --- Metrics now use the reliable 'report_date' column ---
        min_date = repo_lca_data['report_date'].min()
        max_date = repo_lca_data['report_date'].max()
        age_years = ((max_date - min_date).days) / 365.25
        median_bug_year = repo_lca_data['report_date'].dt.year.median()

        # d) No. of authors & e) No. of commits (Correctly scoped to the snapshot)
        all_commits = list(repo.iter_commits())
        num_commits = len(all_commits)
        num_authors = len({c.author.email for c in all_commits})

        # a) LoC & g) Polyglot Index
        loc, polyglot_index = calculate_loc_and_polyglot(repo_path, language)

        # f) No. of external dependencies
        dependencies = count_dependencies(repo_path, language)

        # h) Bug density
        kloc = loc / 1000
        bug_density = (kloc / unique_bugs) if unique_bugs > 0 else 0

        # i) code_complexity
        code_complexity = calculate_average_complexity(repo_path, language)

        # j) Calculate bug report verbosoty
        bug_report_verbosity = repo_lca_data['bug_report'].str.split().str.len().mean()
        
        results.append({
            'repo_name': repo_name,
            'language': language,
            'LoC': loc,
            'age_years': age_years,
            'median_bug_year': median_bug_year, # Raw data for later categorization
            'num_authors': num_authors,
            'num_commits': num_commits,
            'num_dependencies': dependencies,
            'polyglot_index': polyglot_index,
            'bug_density': bug_density,
            'code_complexity': code_complexity,
            'bug_report_verbosity': bug_report_verbosity
        })
    
    # except git.exec.GitCommandError as e:
    #     print(f"Error processing Git Repo {repo_name}: {e}")
    except Exception as e:
        print(f"An unexpected error occurred for {repo_name}: {e}")

if not results:
    print("No results were generated.")
    exit(0)

# Create final DataFrame
final_df = pd.DataFrame(results)

# # b) Calculate project_size category
# final_df = final_df.groupby('language', group_keys=False).apply(categorize_by_tercile)

# Save to CSV
final_df.to_csv(OUTPUT_CSV_PATH, index=False)
print(f"Metdata extraction complete. Results saved to '{OUTPUT_CSV_PATH}'")


Starting metadata extraction for LCA Dataset...


Processing Repos:   0%|          | 0/13 [00:00<?, ?it/s]

Calculating meta data for repo: quarkusio/quarkus


Processing Repos:   8%|▊         | 1/13 [00:43<08:37, 43.09s/it]


Calculating meta data for repo: hannah-sten/texify-idea


...


Calculating meta data for repo: iterative/dvc


Processing Repos:  92%|█████████▏| 12/13 [01:13<00:03,  3.38s/it]


Calculating meta data for repo: rucio/rucio


Processing Repos: 100%|██████████| 13/13 [01:16<00:00,  5.90s/it]


Metdata extraction complete. Results saved to '/home/user/CS21D002_A_Eashaan_Rao/Research/PhD/Objective1/benchmark_dataset_analysis/lca_metadata.csv'
